# Fractional cover workflow (TERN STAC API)
Load items, reduce over an ROI, and plot a time series.


## Before you run
- Update placeholder values (`COLLECTION_ID`, dates, bounds, point coordinates) to match your data.
- Ensure auth is configured for protected assets (for example `.netrc` and/or GDAL config).
- Install optional dependencies required by this notebook's workflow (`odc-stac`, `rioxarray`, `geopandas`, plotting extras).
- Run cells from top to bottom so variables are initialized in order.


In [ ]:
from itertools import islice
from pathlib import Path

from tern_stac import TernStacClient, load_items_as_time_series, plot_time_series


In [ ]:
# Fill in from your catalog values
COLLECTION_ID = "gov_qld_fractional_cover_v3_landsat_fractional_cover__seasonal"
START_DATE = "2020-01-01"
END_DATE = "2025-01-01"

REGION_BOUNDS = (152.914613, -27.561273, 153.142615, -27.367202)  # (minx, miny, maxx, maxy)
REGION_BOUNDS_CRS = "EPSG:4326"


In [ ]:
Path("outputs").mkdir(exist_ok=True)

client = TernStacClient()
search = client.search(
    collections=[COLLECTION_ID],
    datetime=f"{START_DATE}/{END_DATE}",
    bbox=[REGION_BOUNDS[0], REGION_BOUNDS[1], REGION_BOUNDS[2], REGION_BOUNDS[3]],
)
items = list(islice(search.items(), 10))
len(items)


In [ ]:
if not items:
    raise RuntimeError("No items returned. Update collection/date/bbox placeholders.")

ds = load_items_as_time_series(
    items,
    media_type=None,
    role="data",
    chunks=True,
    clip_bounds=REGION_BOUNDS,
    clip_bounds_crs=REGION_BOUNDS_CRS,
    to_numpy_nodata=True,
)
ds


In [ ]:
plot_time_series(
    ds,
    band_dim="band",
    figsize=(12, 6),
    compute=True,
    save_path="outputs/fractional_cover_time_series.png",
    title="Fractional cover ROI mean by band",
)
